# DLPFC 多切片整合


In [ ]:
from pathlib import Path
import sys
import warnings
warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import torch
from sklearn.cluster import KMeans
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import adjusted_rand_score,normalized_mutual_info_score

cwd = Path.cwd().resolve()
PROJECT_ROOT = next(
    (path for path in (cwd, *cwd.parents) if (path / "SpaDiff_improved").is_dir()),
    cwd,
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import SpaDiff_improved as sd
from SpaDiff_improved.utils import mclust_R, set_seed


## 参数

In [ ]:
SEED = 42
ST_SAMPLES = ["151673", "151674", "151675", "151676"]
SAMPLE_ID = "7376"
BATCH_KEY = "batch_name"
REFERENCE_BATCH = ST_SAMPLES[0]
N_CLUSTERS = 7
K_INTRA = 6
K_INTER = 2
N_NEIGHBORS = 10
FAST_DEV_RUN = False

# loss = K1 * original DEC-KL + K2 * diffusion DSM
K1 = 1.0
K2 = 0.0
DEC_ONLY = K1 > 0.0 and K2 == 0.0
CLUSTER_METHOD = "mclust" if DEC_ONLY else "kmeans"

DATA_ROOT = Path("E:/gxy_2/final/0_data/case1")
print("DATA_ROOT =", DATA_ROOT)

set_seed(SEED)
torch.backends.cudnn.deterministic = True
device = torch.device(
    "cuda:0" if torch.cuda.is_available() else "cpu"
)
print("device =", device)
print(f"loss = {K1} * original_DEC + {K2} * diffusion_DSM")


## 读取并拼接四个 Visium 切片

In [ ]:
ST_SAMPLES = ["151673", "151674", "151675", "151676"]
slices = []
for index, sample in enumerate(ST_SAMPLES):
    sample_dir = DATA_ROOT / sample
    current = sc.read_visium(sample_dir)
    current.var_names_make_unique()
    current.layers["counts"] = current.X.copy()
    sc.pp.normalize_total(current, target_sum=1e4)
    sc.pp.log1p(current)

    truth = pd.read_csv(sample_dir / "truth.txt", sep="\t", header=None, index_col=0)
    truth.columns = ["Truth"]
    current.obs["Truth"] = truth.reindex(current.obs_names)["Truth"]
    current.obs[BATCH_KEY] = sample
    current.obs_names = [f"{sample}:{barcode}" for barcode in current.obs_names]
    slices.append(current)

adata = sc.concat(slices, join="inner", merge="same")
# adata.layers["counts"] = adata.X.copy()
sc.pp.highly_variable_genes(adata, flavor="seurat_v3", layer="counts", n_top_genes=5000,batch_key=BATCH_KEY, subset=True)
# sc.pp.scale(adata, max_value=10)
n_components = 50
sc.tl.pca(adata, n_comps=n_components, svd_solver="arpack")
adata


## 构建切片内与切片间高阶拓扑

In [ ]:
coordinates, adjacency = sd.Neiber(adata, k_intra=K_INTRA, k_inter=K_INTER, slice_order=ST_SAMPLES)

adjacency = adjacency.maximum(adjacency.T)
operators = sd.to_torch_operators(sd.build_simplicial_operators(adjacency, max_order=2), device=device)

features = torch.as_tensor(adata.obsm["X_pca"], dtype=torch.float32, device=device)
batch_category = pd.Categorical(adata.obs[BATCH_KEY], categories=ST_SAMPLES, ordered=True)
if np.any(batch_category.codes < 0):
    raise ValueError("存在未包含在 ST_SAMPLES 中的 batch 标签")
batch_ids = torch.as_tensor(batch_category.codes, dtype=torch.long, device=device)
modality_ids = torch.zeros(adata.n_obs, dtype=torch.long, device=device)
print("features:", tuple(features.shape), "batches:", batch_category.categories.tolist())
print("edge nnz:", operators[1]._nnz(), "triangle nnz:", operators[2]._nnz())


## 训练 batch 条件 VP-SDE

In [ ]:
config = sd.SpaDiffConfig(
    data_dim=features.shape[1],
    condition_input_dim=features.shape[1],
    num_batches=len(ST_SAMPLES),
    num_modalities=1,
    num_scales=1000,
    topology_hidden_dim=64,
    topology_dim=256,
    propagation_steps= 5,
    propagation_alpha= 0.4,
    hidden_dim=128,
    dropout=0.2,
    topology_projection_dropout=0.0,
    k1=K1,
    k2=K2,
    num_clusters=N_CLUSTERS,
    dec_alpha=1.0,
    dec_update_interval=10,
    dec_tolerance=1e-4,
    dec_init_method="mclust",
    random_seed=SEED,
)
model = sd.SpaDiff(config).to(device)
training_epochs = 200
training = sd.train_spadiff(
    model, features, operators, batch_ids, modality_ids,
    epochs=training_epochs,
    learning_rate=1e-3,
    weight_decay=1e-4,
    ema_decay=None if DEC_ONLY else 0.999,
    verbose_every=1 if FAST_DEV_RUN or DEC_ONLY else 25,
)
print("best total loss =", training.best_loss)
print("last original DEC loss =", training.original_losses[-1])
print("last diffusion loss =", training.diffusion_losses[-1])


In [ ]:
if DEC_ONLY:
    dec_labels, dec_prob, embedding = model.original_predict(features, operators)
    corrected = embedding
else:
    reference_code = ST_SAMPLES.index(REFERENCE_BATCH)
    reference_ids = torch.full_like(batch_ids, reference_code)
    if training.ema is not None:
        training.ema.store(model.parameters())
        training.ema.copy_to(model.parameters())
    try:
        corrected = model.harmonize(
            observed_features=features,
            operators=operators,
            reference_batch_ids=reference_ids,
            modality_ids=modality_ids,
            strength=0.35,
            sampler="ode",
            guidance_scale=1.1,
            ode_steps=20 if FAST_DEV_RUN else 300,
        )
    finally:
        if training.ema is not None:
            training.ema.restore(model.parameters())

adata.obsm["spadiff"] = corrected.cpu().numpy()


## 空间域聚类

In [ ]:

labels = mclust_R(adata, num_cluster=N_CLUSTERS, used_obsm="spadiff",pca_num=15)

adata.obs["mclust"] = pd.Categorical(labels.astype(str))

ari_by_slice = {}
nmi_by_slice = {}

for sample in ST_SAMPLES:
    current = adata.obs.loc[adata.obs[BATCH_KEY] == sample, ["Truth", "mclust"]].dropna()
    ari_by_slice[sample] = adjusted_rand_score(current["Truth"], current["mclust"])
    nmi_by_slice[sample] = normalized_mutual_info_score(current["Truth"], current["mclust"])
valid = adata.obs[["Truth", "mclust"]].dropna()

overall_ari = round(adjusted_rand_score(valid["Truth"], valid["mclust"]),4)
overall_nmi = round(normalized_mutual_info_score(valid["Truth"],valid["mclust"]),4)

print("ARI by slice:", {key: round(value, 3) for key, value in ari_by_slice.items()})
print("NMI by slice:",{key: round(value, 3) for key, value in nmi_by_slice.items()})

print(f"overall ARI = {overall_ari:.3f}")
print(f"overall NMI = {overall_nmi:.3f}")


In [ ]:
palette = ["#6D1A9C", "#D1D1D1", "#F56867", "#59BE86", "#FEB915", "#C798EE", "#7495D3"]
fig, axes = plt.subplots(1, len(ST_SAMPLES), figsize=(5 * len(ST_SAMPLES), 5))
for axis, sample in zip(axes, ST_SAMPLES):
    current = adata[adata.obs[BATCH_KEY] == sample].copy()
    sc.pl.spatial(
        current, color="mclust", ax=axis, show=False, spot_size=120,
        palette=palette, legend_loc=None, title=f"{sample} | ARI={ari_by_slice[sample]:.3f}"
    )
mode = "DEC only" if DEC_ONLY else f"hybrid k1={K1:g}, k2={K2:g}"
fig.suptitle(f"SpaDiff {mode} | overall ARI={overall_ari:.3f}", fontsize=16)
plt.tight_layout()
# plt.show()
# plt.savefig("../result/spadiff_"+SAMPLE_ID+"_"+str(overall_ari)+".pdf", bbox_inches='tight')

In [ ]:
# sc.pp.neighbors(adata, use_rep="spadiff")
# sc.tl.umap(adata, random_state=SEED)
# sc.pl.umap(adata, color=[BATCH_KEY, "Truth", "mclust"], wspace=0.35)


In [ ]:
for i in range(12,25):
    print(f"pca:{i}")
    labels = mclust_R(adata, num_cluster=N_CLUSTERS, used_obsm="spadiff", pca_num= i)
    adata.obs["mclust"] = pd.Categorical(labels.astype(str))
    
    valid = adata.obs[["Truth", "mclust"]].dropna()
    ari = round(adjusted_rand_score(valid["Truth"], valid["mclust"]),4)
    print(ari)
    
    fig, axes = plt.subplots(1, len(ST_SAMPLES), figsize=(5 * len(ST_SAMPLES), 5))
    for axis, sample in zip(axes, ST_SAMPLES):
        current = adata[adata.obs[BATCH_KEY] == sample].copy()
        sc.pl.spatial(
            current, color="mclust", ax=axis, show=False, spot_size=120,
            palette=palette, legend_loc=None, title=""
        )
    fig.suptitle(f"SpaDiff| overall ARI={ari:.3f}", fontsize=16)
    plt.tight_layout()
    plt.show()

    if ari >= 0.64:
        break

In [ ]:
# output_file ="../result/spadiff_"+SAMPLE_ID+"_"+str(overall_ari)+".h5ad"  # 压缩后的 h5ad 文件路径

# adata.write_h5ad(output_file, compression="gzip")